# Consolidated Hyperliquid analytics implementation

Methods used by `hyperliquid_multi_account_analytics.ipynb`. It reuses the single-wallet implementation and combines data from each configured wallet.


## Shared aggregation helpers

Aggregate historical account value over dates available for every wallet, and prepare a combined live snapshot and realized/funding history.


In [ ]:
def _combine_wallet_data(wallet_analytics):
    """Combine account inputs while keeping the full common history window."""
    account_series = []
    realized_frames = []
    fills = []
    funding_frames = []
    for wallet, analytics in wallet_analytics.items():
        data = analytics.data
        chart = data.chart_df.copy()
        chart['ts'] = pd.to_datetime(chart['ts'], utc=True).dt.tz_convert(ACCOUNTING_TIMEZONE)
        history = chart.loc[chart['source'] == 'portfolio_allTime'].set_index('ts')['accountValue'].dropna()
        daily = history.resample('1D').last().dropna()
        if not daily.empty:
            account_series.append(daily.rename(wallet))
        realized = chart.loc[chart['source'].isin([
            'perp_realized_daily','spot_realized_daily','funding_daily'
        ])].copy()
        if not realized.empty:
            realized['wallet'] = wallet
            realized['ts'] = realized['ts'].dt.normalize()
            realized_frames.append(realized)
        frame = data.realized_df.copy()
        if not frame.empty:
            frame['wallet'] = wallet
            fills.append(frame)
        funding = data.funding_history_df.copy()
        if not funding.empty:
            funding['wallet'] = wallet
            funding_frames.append(funding)

    if account_series:
        daily_accounts = pd.concat(account_series, axis=1, join='inner').dropna()
        if daily_accounts.empty:
            raise ValueError('The configured wallets have no overlapping all-time account history.')
        account_history = daily_accounts.sum(axis=1).rename('accountValue').rename_axis('ts').reset_index()
        account_history['source'] = 'portfolio_allTime'
        for column in ['balance','open_position_value','unrealized_pnl','realized_pnl','asset']:
            account_history[column] = float('nan') if column != 'asset' else ''
        account_history = account_history[HISTORY_COLUMNS]
    else:
        account_history = pd.DataFrame(columns=HISTORY_COLUMNS)

    if realized_frames:
        realized_history = pd.concat(realized_frames, ignore_index=True)
        realized_history['asset'] = realized_history['asset'].fillna('')
        realized_history = realized_history.groupby(
            ['ts','source','asset'],as_index=False,dropna=False
        )['realized_pnl'].sum()
        for column in ['accountValue','balance','open_position_value','unrealized_pnl']:
            realized_history[column] = float('nan')
        realized_history = realized_history[HISTORY_COLUMNS]
    else:
        realized_history = pd.DataFrame(columns=HISTORY_COLUMNS)

    total_balance = sum(analytics.data.balance_total for analytics in wallet_analytics.values())
    total_unrealized = sum(analytics.data.unrealized_pnl_total for analytics in wallet_analytics.values())
    total_position_value = sum(analytics.data.position_value_total for analytics in wallet_analytics.values())
    current_value = total_balance + total_unrealized
    snapshot = pd.DataFrame([{
        'ts':pd.Timestamp.now(tz=ACCOUNTING_TIMEZONE),
        'accountValue':current_value,'source':'live_snapshot','asset':'',
        'balance':total_balance,'open_position_value':total_position_value,
        'unrealized_pnl':total_unrealized,'realized_pnl':float('nan'),
    }])[HISTORY_COLUMNS]
    chart_df = pd.concat([account_history,realized_history,snapshot],ignore_index=True)
    chart_df['ts'] = pd.to_datetime(chart_df['ts'])
    chart_df = chart_df.sort_values('ts').reset_index(drop=True)
    realized_fills = pd.concat(fills,ignore_index=True) if fills else pd.DataFrame()
    if not realized_fills.empty:
        realized_fills['ts'] = pd.to_datetime(realized_fills['ts'],utc=True)
        realized_fills['day'] = pd.to_datetime(realized_fills['day'],utc=True)
    now_et=pd.Timestamp.now(tz=ACCOUNTING_TIMEZONE)
    cutoff=now_et.normalize()
    today_fill_pnl=0.0
    if not realized_fills.empty:
        today_fill_pnl=float(realized_fills.loc[realized_fills['ts']>=cutoff,'closedPnl'].sum())
    today_funding_pnl=0.0
    for analytics in wallet_analytics.values():
        raw_funding=analytics.data.funding_df
        if raw_funding is not None and not raw_funding.empty:
            funding_ts=pd.to_datetime(raw_funding['ts'],utc=True)
            today_funding_pnl+=float(
                raw_funding.loc[funding_ts>=cutoff,'fundingPnl'].sum()
            )
    today_realized_et=today_fill_pnl+today_funding_pnl
    funding_history = pd.concat(funding_frames,ignore_index=True) if funding_frames else pd.DataFrame(
        columns=['ts','coin','fundingPnl','wallet']
    )
    if not funding_history.empty:
        funding_history['ts'] = pd.to_datetime(funding_history['ts'],utc=True)
        funding_history = funding_history.groupby(['ts','coin'],as_index=False)['fundingPnl'].sum()
    return SimpleNamespace(
        chart_df=chart_df,history_rows_added=sum(a.data.history_rows_added for a in wallet_analytics.values()),
        realized_fills=sum((a.data.realized_fills for a in wallet_analytics.values()),[]),
        realized_df=realized_fills,funding_history_df=funding_history,
        current_account_value=current_value,balance_total=total_balance,
        unrealized_pnl_total=total_unrealized,position_value_total=total_position_value,
        realized_1d=today_realized_et,
    )


def _display_consolidated_positions(rows):
    if not rows:
        print('No open positions across the configured wallets.')
        return
    frame=pd.DataFrame(rows).sort_values('Pos',key=lambda values:values.abs(),ascending=False).reset_index(drop=True)
    columns=['Wallet','Coin','Side','Liq %','ROE','uPnL','Funding','Funding (1h)','Funding APR',
             'Entry','Mark','Pos','Margin','Margin Type','Margin %','Fees','Liq Px','Lev.','Sz']
    usd=lambda value:f'-${abs(value):,.2f}' if value<0 else f'${value:,.2f}'
    def style(row_index,column,value,row):
        if column=='Side' and value in {'LONG','SHORT'}:
            return 'color:#188038;font-weight:600;' if value=='LONG' else \
                'color:#c62828;font-weight:600;'
        if column=='Mark' and pd.notna(value) and pd.notna(row['Entry']):
            out_of_money=(row['Side']=='LONG' and value<row['Entry']) or \
                (row['Side']=='SHORT' and value>row['Entry'])
            return 'color:#c62828;font-weight:600;' if out_of_money else ''
        if column in {'Funding (1h)','Funding APR'} and pd.notna(value):
            if value==0: return ''
            pays=(value>0 and row['Side']=='LONG') or (value<0 and row['Side']=='SHORT')
            return 'color:#c62828;font-weight:600;' if pays else 'color:#188038;font-weight:600;'
        if column == 'Liq %' and pd.notna(value):
            return 'color:#c62828;' if value <= 10 else \
                'color:#d97706;' if value <= 25 else 'color:#188038;'
        if column in {'uPnL','ROE','Funding'} and pd.notna(value):
            return 'color:#188038;' if value>0 else 'color:#c62828;' if value<0 else ''
        if column=='Fees': return 'color:#c62828;'
        return ''
    display_interactive_table(frame,columns=columns,cell_style=style,formatters={
        'Sz':lambda v:f'{v:,.2f}','Pos':lambda v:f'${v:,.0f}','Margin':lambda v:f'${v:,.0f}',
        'Liq %':lambda v:f'{v:.2f}%' if pd.notna(v) else 'n/a',
        'ROE':lambda v:f'{v:.2f}%','uPnL':usd,'Funding':usd,
        'Funding (1h)':lambda v:f'{v:.6f}%' if pd.notna(v) else 'n/a',
        'Funding APR':lambda v:f'{v:.2f}%' if pd.notna(v) else 'n/a',
        'Entry':lambda v:f'{v:,.8g}','Mark':lambda v:f'{v:,.8g}',
        'Liq Px':lambda v:f'{v:,.5g}' if pd.notna(v) else 'n/a','Lev.':lambda v:f'{v:.2f}x',
        'Margin %':lambda v:f'{v:.2f}%' if pd.notna(v) else 'n/a',
    })


In [ ]:
def _fetch_daily_atr(coin, period):
    if not isinstance(period, int) or period <= 0:
        raise ValueError('ATR_PERIOD must be a positive integer.')
    now = pd.Timestamp.now(tz='UTC')
    start = now - pd.Timedelta(days=max(period * 10 + 30, 365))
    candles = pd.DataFrame(post({
        'type':'candleSnapshot',
        'req':{
            'coin':coin,'interval':'1d',
            'startTime':int(start.timestamp() * 1000),
            'endTime':int(now.timestamp() * 1000),
        },
    }))
    required = {'t','h','l','c'}
    if candles.empty or not required.issubset(candles.columns):
        raise RuntimeError(f'No usable daily candles returned for {coin}; cannot calculate ATR.')
    candles = candles.copy()
    candles['t'] = pd.to_numeric(candles['t'], errors='coerce')
    candles = candles.loc[candles['t'] < now.normalize().timestamp() * 1000].sort_values('t')
    for column in ['h','l','c']:
        candles[column] = pd.to_numeric(candles[column], errors='coerce')
    candles = candles.dropna(subset=['t','h','l','c'])
    previous_close = candles['c'].shift(1)
    true_range = pd.concat([
        candles['h'] - candles['l'],
        (candles['h'] - previous_close).abs(),
        (candles['l'] - previous_close).abs(),
    ],axis=1).max(axis=1).dropna()
    if true_range.empty:
        raise RuntimeError(f'No completed daily true-range observations available for {coin}.')
    period_used = min(period, len(true_range))
    atr = float(true_range.iloc[:period_used].mean())
    for value in true_range.iloc[period_used:]:
        atr = (atr * (period_used - 1) + float(value)) / period_used
    if not np.isfinite(atr) or atr <= 0:
        raise RuntimeError(f'Invalid ATR({period}) calculated for {coin}: {atr!r}.')
    return atr, period_used


def _display_stop_sizing(rows, risk_budget_pct, account_value,
                         atr_period, atr_stop_multiple):
    if not rows:
        return
    if not np.isfinite(risk_budget_pct) or risk_budget_pct <= 0:
        raise ValueError('RISK_BUDGET_PCT must be a positive finite percentage.')
    if not np.isfinite(atr_stop_multiple) or atr_stop_multiple <= 0:
        raise ValueError('ATR_STOP_MULTIPLE must be a positive finite value.')
    if not np.isfinite(account_value) or account_value <= 0:
        raise ValueError('Combined account value must be positive to calculate risk-based sizing.')
    grouped = {}
    for row in rows:
        key = (str(row['Coin']), str(row['Side']).upper())
        if key[1] not in {'LONG', 'SHORT'}:
            raise ValueError(f'Unsupported position side for {key[0]}: {key[1]}')
        grouped.setdefault(key, []).append(row)
    sides_by_asset = {}
    for coin, side in grouped:
        sides_by_asset.setdefault(coin, set()).add(side)
    output = []
    atr_by_asset = {}
    for (coin, side), positions in sorted(grouped.items()):
        direction = 1 if side == 'LONG' else -1
        quantities = [abs(float(position['Sz'])) for position in positions]
        entries = [float(position['Entry']) for position in positions]
        marks = [float(position['Mark']) for position in positions]
        total_quantity = sum(quantities)
        if total_quantity <= 0 or not all(np.isfinite(value) for value in quantities + entries + marks):
            raise ValueError(f'Cannot calculate position sizing for {coin} {side}: invalid size, entry, or mark.')
        average_entry = sum(quantity * entry for quantity, entry in zip(quantities, entries)) / total_quantity
        current_mark = sum(quantity * mark for quantity, mark in zip(quantities, marks)) / total_quantity
        if coin not in atr_by_asset:
            atr_by_asset[coin] = _fetch_daily_atr(coin, atr_period)
        atr, atr_period_used = atr_by_asset[coin]
        stop_distance = atr_stop_multiple * atr
        stop = current_mark - direction * stop_distance
        asset_risk_budget = account_value * risk_budget_pct / 100.0
        side_count = len(sides_by_asset[coin])
        side_risk_budget = asset_risk_budget / side_count
        target_quantity = side_risk_budget / stop_distance
        take_profit = current_mark + direction * stop_distance
        open_risk = sum(
            quantity * max(direction * (mark - stop), 0.0)
            for quantity, mark in zip(quantities, marks)
        )
        size_difference = direction * (total_quantity - target_quantity)
        size_difference_pct = (total_quantity - target_quantity) / target_quantity * 100.0
        output.append({
            'Asset': coin, 'Side': side, 'ATR (%)': atr / current_mark * 100.0,
            'ATR Period Used': atr_period_used,
            'ATR Stop Distance (%)': stop_distance / current_mark * 100.0,
            'Avg Entry': average_entry, 'Mark': current_mark, 'Stop': stop,
            'Take Profit': take_profit,
            'Open - Target (coins)': size_difference,
            'Open - Target (%)': size_difference_pct,
            'Open Risk ($)': open_risk,
        })
    frame = pd.DataFrame(output)
    print(f'Fixed-fractional sizing: risk budget {risk_budget_pct:.2f}% of combined account value '
          f'per asset; stop distance {atr_stop_multiple:g} x daily ATR({atr_period}). '
          'Opposing sides, when both open, split the asset budget equally.')
    short_history = frame.loc[frame['ATR Period Used'] < atr_period, 'Asset'].unique()
    if len(short_history):
        print(f'Limited daily history; ATR uses all completed candles available (shorter than '
              f'{atr_period} days) for: {", ".join(short_history)}.')
    print('ATR stop-distance colors: green <2%, orange 2-5%, red >5%; red mark means '
          'the position is out of the money; stop prices are highlighted red.')
    def sizing_cell_style(row_index, column, value, row):
        if column == 'Stop':
            return 'color:#c62828;font-weight:600;'
        if column == 'Mark':
            out_of_money = (row['Side'] == 'LONG' and value < row['Avg Entry']) or (
                row['Side'] == 'SHORT' and value > row['Avg Entry']
            )
            return 'color:#c62828;font-weight:600;' if out_of_money else ''
        if column == 'ATR Stop Distance (%)':
            if value < 2:
                return 'color:#188038;font-weight:600;'
            if value <= 5:
                return 'color:#b06000;font-weight:600;'
            return 'color:#c62828;font-weight:600;'
        return ''
    display_interactive_table(frame, formatters={
        'ATR (%)':lambda value:f'{value:.2f}%',
        'ATR Period Used':lambda value:f'{int(value)} days',
        'ATR Stop Distance (%)':lambda value:f'{value:.2f}%',
        'Avg Entry':lambda value:f'{value:,.8g}',
        'Mark':lambda value:f'{value:,.8g}',
        'Stop':lambda value:f'{value:,.5g}',
        'Take Profit':lambda value:f'{value:,.5g}',
        'Open - Target (coins)':lambda value:f'{value:,.6g}' if pd.notna(value) else 'n/a',
        'Open - Target (%)':lambda value:f'{value:,.2f}%' if pd.notna(value) else 'n/a',
        'Open Risk ($)':lambda value:f'${value:,.2f}' if pd.notna(value) else 'n/a',
    }, cell_style=sizing_cell_style)



## Wallet data collection

Dashboard call: `analytics.fetch_data()`. Fetch each wallet once, combine daily account history over the shared date range, and pool fills and funding.


In [ ]:
class _MultiWalletDataMixin:
    def fetch_data(self, refresh_full_history=False):
        self.wallet_analytics={}
        for wallet in self.wallet_addresses:
            account=HyperliquidAnalytics(wallet,self.data_dir)
            account.fetch_data(refresh_full_history=refresh_full_history)
            self.wallet_analytics[wallet]=account
        self.data=_combine_wallet_data(self.wallet_analytics)
        self.account_metrics=calculate_account_metrics(
            self.data.chart_df,self.data.current_account_value,self.data.unrealized_pnl_total
        )
        self.positions=[]
        self._positions_loaded=False
        fill_count=sum(len(a.data.realized_fills) for a in self.wallet_analytics.values())
        print(f'Loaded {len(self.wallet_addresses)} wallets; {fill_count:,} available fills; '
              f'combined current account value ${self.data.current_account_value:,.2f}.')
        common=self.data.chart_df.loc[self.data.chart_df['source']=='portfolio_allTime','ts']
        if not common.empty:
            print(f'Combined historical account value uses shared daily coverage in '
                  f'{ACCOUNTING_TIMEZONE} from {common.min().date()} through {common.max().date()}.')
        return self.data

    def _require_data(self):
        if self.data is None:
            raise RuntimeError('Call fetch_data() before requesting a consolidated report.')
        return self.data

    def _load_positions(self):
        self._require_data()
        if not self._positions_loaded:
            self.positions=[]
            for wallet,account in self.wallet_analytics.items():
                rows=account._load_positions()
                for row in rows:
                    self.positions.append({**row,'Wallet':wallet[-4:]})
            self._positions_loaded=True
        return self.positions


## Account reports

Dashboard calls mirror the single-wallet account, chart, and BTC comparison sections; values and time series are consolidated.


In [ ]:
class _MultiWalletAccountMixin:
    def show_account_pnl(self):
        data=self._require_data()
        print_account_metrics(
            self.account_metrics,data.history_rows_added,len(data.realized_fills),
            f'${data.current_account_value:,.2f} (combined live wallet value)',
            realized_1d=data.realized_1d,
        )

    def show_account_charts(self):
        self._require_data()
        plot_account_dashboard(self.account_metrics)

    def show_account_vs_btc(self):
        data=self._require_data()
        plot_value_added_indexes(self.account_metrics,data.chart_df,data.current_account_value)


## Open positions

Dashboard call: `analytics.show_open_positions()`. Show per-wallet positions, stop-based sizing grouped by asset/side, and an aggregate account summary.


In [ ]:
class _MultiWalletPositionsMixin:
    def show_open_positions(self):
        data=self._require_data()
        rows=self._load_positions()
        _display_consolidated_positions(rows)
        _display_stop_sizing(
            rows,self.position_risk_budget_pct,
            data.current_account_value,self.position_atr_period,self.position_atr_stop_multiple,
        )
        position_value=sum(abs(row['Pos']) for row in rows)
        margin=sum(row['Margin'] for row in rows)
        funding=sum(row['Funding'] for row in rows)
        summary=pd.DataFrame([
            ('Wallets',f'{len(self.wallet_addresses):,}'),
            ('Position value',f'${position_value:,.0f}'),
            ('uPnL',f'${data.unrealized_pnl_total:,.2f}'),
            ('ROE',f'{data.unrealized_pnl_total/margin*100:,.2f}%' if margin else 'n/a'),
            ('Margin',f'${margin:,.0f}'),
            ('Margin % bal',f'{margin/data.balance_total*100:,.2f}%' if data.balance_total else 'n/a'),
            ('Leverage (Pos / Margin)',f'{position_value/margin:,.2f}x' if margin else 'n/a'),
            ('Funding',f'${funding:,.2f}'),
            ('Account value',f'${data.current_account_value:,.2f}'),
        ],columns=['Metric','Value'])
        def style(*cell):
            _,column,value,row=cell
            if column!='Value':
                return ''
            if row['Metric'] in {'Margin','Position value','Account value'}:
                return 'color:#188038;font-weight:600;'
            if row['Metric'] not in {'uPnL','ROE','Funding'}:
                return ''
            if isinstance(value,str) and (value.startswith('-') or '$-' in value):
                return 'color:#c62828;font-weight:600;'
            if isinstance(value,str) and value!='n/a':
                return 'color:#188038;font-weight:600;'
            return ''
        display_interactive_table(summary,cell_style=style)
        print('ROE is unrealized PnL divided by used margin; a negative ROE means the open positions '
              'are currently at an unrealized loss relative to the margin supporting them.')


## Realized PnL

Dashboard call: `analytics.show_realized_pnl()`. Pool fills and funding from all configured wallets.


In [ ]:
class _MultiWalletRealizedMixin:
    def show_realized_pnl(self):
        data=self._require_data()
        perp_df,spot_df=split_realized_fills(data.realized_df)
        print_realized_stats(spot_df,'SPOT',account_value=data.current_account_value)
        print_realized_stats(perp_df,'PERP',funding_history=data.funding_history_df,
                             account_value=data.current_account_value)
        print('\n=== PERP PNL BREAKDOWN ===')
        breakdown,non_usd_tokens=perp_pnl_breakdown(perp_df,data.funding_history_df)
        net_index=breakdown.index[breakdown['Component']=='Net perp PnL'][0]
        display_interactive_table(breakdown,formatters={
            'Amount ($)':lambda v:f'${v:,.2f}',
            'Contribution (% of |net PnL|)':lambda v:f'{v:.2f}%',
        },cell_style=breakdown_style,bold_last=True,
          blank_missing_cells={(net_index,'Contribution (% of |net PnL|)')})
        liquidations=liquidation_stats(perp_df,funding_history=data.funding_history_df)
        print(f"Liquidations: {liquidations['liquidations']} of {liquidations['closes']} closes "
              f"({liquidations['close_pct']:.2f}%) | liquidation PnL: ${liquidations['pnl']:,.2f}.")
        if non_usd_tokens:
            print(f'Trading/fee dollar contributions unavailable for fee token(s): {non_usd_tokens}.')


## Volume reports

Dashboard calls: `analytics.show_volume_chart()` and `analytics.show_volume_summary()`. Pooled fill data feeds both outputs.


In [ ]:
class _MultiWalletVolumeMixin:
    def show_volume_chart(self):
        data=self._require_data()
        plot_volume_pnl(data.realized_df)

    def show_volume_summary(self):
        data=self._require_data()
        summary=summarize_volume_pnl(build_daily_volume_pnl(data.realized_df))
        print('=== TRADED VOLUME VS REALIZED PNL (ALL WALLETS) ===')
        if summary.empty:
            print('No fill volume or realized PnL data is available.')
            return
        def style(row_index,column,value,row):
            if column in {'Realized PnL ($)','Realized PnL / Volume (%)','PnL / $1k volume ($)'} and pd.notna(value):
                return 'color:#188038;' if value>0 else 'color:#c62828;' if value<0 else ''
            return ''
        display_interactive_table(summary,formatters={
            'Volume ($)':lambda v:f'${v:,.2f}','Realized PnL ($)':lambda v:f'${v:,.2f}',
            'Realized PnL / Volume (%)':lambda v:f'{v:.2f}%',
            'PnL / $1k volume ($)':lambda v:f'${v:,.2f}',
            'Daily Pearson r':lambda v:f'{v:.3f}','Daily Spearman r':lambda v:f'{v:.3f}',
        },cell_style=style)


## PnL attribution

Dashboard call: `analytics.show_pnl_attribution()`. Combine realized PnL, open positions, and funding by asset across wallets.


In [ ]:
class _MultiWalletAttributionMixin:
    def show_pnl_attribution(self):
        data=self._require_data()
        positions=self._load_positions()
        attribution,net_pnl=build_pnl_attribution(
            data.realized_df,positions,funding_history=data.funding_history_df
        )
        print('=== CONSOLIDATED PNL ATTRIBUTION ===')
        if attribution.empty:
            print('No realized or open-position PnL to attribute.')
            return
        def style(row_index,column,value,row):
            if column in {'Realized','uPnL','Total PnL','Realized (% gross)',
                          'uPnL (% gross)','Total (% gross)'} and pd.notna(value):
                return 'color:#188038;' if value>0 else 'color:#c62828;' if value<0 else ''
            return ''
        display_interactive_table(attribution,formatters={
            'Closes':lambda v:f'{v:,.0f}','Realized':lambda v:f'${v:,.2f}',
            'uPnL':lambda v:f'${v:,.2f}','Total PnL':lambda v:f'${v:,.2f}',
            'Realized (% gross)':lambda v:f'{v:.2f}%','uPnL (% gross)':lambda v:f'{v:.2f}%',
            'Total (% gross)':lambda v:f'{v:.2f}%',
        },cell_style=style,bold_last=True,blank_missing_cells={
            (len(attribution)-1,column) for column in
            ('Realized (% gross)','uPnL (% gross)','Total (% gross)')
        })
        print(f'Net attributed PnL across wallets: ${net_pnl:,.2f}.')


## Consolidated analytics facade

Create one object with the dashboard wallet list. It composes the report methods above.


In [ ]:
class MultiWalletAnalytics(
    _MultiWalletDataMixin,_MultiWalletAccountMixin,_MultiWalletPositionsMixin,
    _MultiWalletRealizedMixin,_MultiWalletVolumeMixin,_MultiWalletAttributionMixin,
):
    def __init__(self,wallet_addresses,data_dir='data'):
        self.wallet_addresses=list(dict.fromkeys(wallet_addresses))
        if not self.wallet_addresses:
            raise ValueError('Configure at least one wallet address.')
        for wallet in self.wallet_addresses:
            if not isinstance(wallet,str) or not re.fullmatch(r'0x[0-9a-fA-F]{40}',wallet):
                raise ValueError(f'Invalid wallet address: {wallet!r}')
        self.data_dir=Path(data_dir)
        self.data_dir.mkdir(parents=True,exist_ok=True)
        self.position_risk_budget_pct=5.0
        self.position_atr_period=14
        self.position_atr_stop_multiple=1.25
        self.data=None
        self.account_metrics=None
        self.positions=[]
        self._positions_loaded=False
